# Libraries :

In [73]:
import numpy as np
from pandas import read_excel
import pandas as pd 
from sklearn.preprocessing import StandardScaler 
import datetime as dt
from tensorflow import keras
from sklearn.preprocessing import OneHotEncoder

# Data Preprocessing :

In [74]:
data = pd.read_excel("Online Retail.xlsx")

In [75]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [76]:
data.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386048,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


In [77]:
data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [78]:
data.isnull().sum().sort_values(ascending=False)

CustomerID     135080
Description      1454
StockCode           0
InvoiceNo           0
Quantity            0
InvoiceDate         0
UnitPrice           0
Country             0
dtype: int64

## remove cancelled orders (negative quantities, InvoiceNo starting with 'C')

In [79]:
data = data[( data["Quantity"]  > 0 ) & (~data["InvoiceNo"].astype(str).str.startswith("C") )]
data = data[data['UnitPrice'] > 0]

In [80]:
data.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,530104.000000,530104,530104.000000,397884.000000
mean,10.542037,2011-07-04 20:16:05.225087744,3.907625,15294.423453
min,1.000000,2010-12-01 08:26:00,0.001000,12346.000000
25%,1.000000,2011-03-28 12:22:00,1.250000,13969.000000
50%,3.000000,2011-07-20 12:58:00,2.080000,15159.000000
75%,10.000000,2011-10-19 12:39:00,4.130000,16795.000000
max,80995.000000,2011-12-09 12:50:00,13541.330000,18287.000000
std,155.524124,NaN,35.915681,1713.141560


## Handle missing CustomerIDs (approximately 135,000 records) 

In [81]:
data = data.dropna(subset=['CustomerID'])
data['Description'] = data['Description'].fillna('unknown')

In [82]:
data.isnull().sum().sort_values(ascending=False)

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

## Calculate Customer Lifetime Value: Total spending per customer (Quantity × UnitPrice)

In [83]:
data['customer lifetime value'] = data['Quantity'] * data['UnitPrice']

## duplicates handled

In [84]:
data.duplicated().sum()

np.int64(5192)

In [85]:
data = data.drop_duplicates()

In [86]:
data.duplicated().sum()

np.int64(0)

## removed outliers

In [87]:
Q1 = data['Quantity'].quantile(0.25)
Q3 = data['Quantity'].quantile(0.75)
IQR = Q3 - Q1
data = data[~((data['Quantity'] < (Q1 - 1.5 * IQR)) | (data['Quantity'] > (Q3 + 1.5 * IQR)))]

In [88]:
Q1 = data['UnitPrice'].quantile(0.25)
Q3 = data['UnitPrice'].quantile(0.75)
IQR = Q3 - Q1
data = data[~((data['UnitPrice'] < (Q1 - 1.5 * IQR)) | (data['UnitPrice'] > (Q3 + 1.5 * IQR)))]

# Create RFM features:

In [89]:
data['Total'] = data['Quantity'] * data['UnitPrice']

refDate = data['InvoiceDate'].max() + dt.timedelta(days=1)

lastDate = data.groupby('CustomerID')['InvoiceDate'].max()
recency = (refDate - lastDate).dt.days

frequency = data.groupby('CustomerID')['InvoiceNo'].nunique()
monetary = data.groupby('CustomerID')['Total'].sum()

rfm = pd.DataFrame({'Recency': recency, 'Frequency': frequency, 'Monetary': monetary})

data = data.merge(rfm, on='CustomerID', how='left')

### Engineer temporal features

In [90]:
data['hour'] = data['InvoiceDate'].dt.hour
data['dayofweek'] = data['InvoiceDate'].dt.dayofweek
data['month'] = data['InvoiceDate'].dt.month

### season from InvoiceDate

In [91]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'
data['season'] = data['month'].apply(get_season)

### Handle categorical variables
  - Country and season

In [92]:
data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,customer lifetime value,Total,Recency,Frequency,Monetary,hour,dayofweek,month,season
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,15.30,372,34,4462.16,8,2,12,Winter
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,20.34,372,34,4462.16,8,2,12,Winter
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,22.00,372,34,4462.16,8,2,12,Winter
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,20.34,372,34,4462.16,8,2,12,Winter
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,20.34,372,34,4462.16,8,2,12,Winter


In [93]:
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_features = ohe.fit_transform(data[['Country', 'season']])


ohe_df = pd.DataFrame(ohe_features, columns=ohe.get_feature_names_out(['Country', 'season']))
data = pd.concat([data.reset_index(drop=True), ohe_df.reset_index(drop=True)], axis=1)
data.drop(columns=['Country', 'season'], inplace=True)
data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,customer lifetime value,Total,Recency,...,Country_Sweden,Country_Switzerland,Country_USA,Country_United Arab Emirates,Country_United Kingdom,Country_Unspecified,season_Fall,season_Spring,season_Summer,season_Winter
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,15.30,15.30,372,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,20.34,20.34,372,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,22.00,22.00,372,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,20.34,20.34,372,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,20.34,20.34,372,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


### Handle categorical variables
 - product categories from Description

In [94]:
keyWords = '(CANDLE|BAG|MUG|HANGER|TOY|CUP|CHRISTMAS|PARTY|STICKER)'

data['Category'] = data['Description'].str.upper().str.extract(keyWords, expand=False)
data['Category'] = data['Category'].fillna('OTHER')

In [95]:
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_features = ohe.fit_transform(data[['Category']])


ohe_df = pd.DataFrame(ohe_features, columns=ohe.get_feature_names_out(['Category']))
data = pd.concat([data.reset_index(drop=True), ohe_df.reset_index(drop=True)], axis=1)
data.drop(columns=['Category'], inplace=True)
data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,customer lifetime value,Total,Recency,...,Category_BAG,Category_CANDLE,Category_CHRISTMAS,Category_CUP,Category_HANGER,Category_MUG,Category_OTHER,Category_PARTY,Category_STICKER,Category_TOY
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,15.30,15.30,372,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,20.34,20.34,372,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,22.00,22.00,372,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,20.34,20.34,372,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,20.34,20.34,372,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


##  Normalize numerical features using StandardScaler 

In [96]:
dataEda = data.copy()

In [97]:
numeric_features = ['Quantity', 'UnitPrice', 'Total', 'hour', 'dayofweek', 'month','Recency', 'Frequency', 'Monetary']

scaler = StandardScaler()
numeric_features_scaled = scaler.fit_transform(data[numeric_features])
data[numeric_features] = numeric_features_scaled

In [98]:
data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,customer lifetime value,Total,Recency,...,Category_BAG,Category_CANDLE,Category_CHRISTMAS,Category_CUP,Category_HANGER,Category_MUG,Category_OTHER,Category_PARTY,Category_STICKER,Category_TOY
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,-0.228808,2010-12-01 08:26:00,0.230129,17850.0,15.30,0.181440,4.996809,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,536365,71053,WHITE METAL LANTERN,-0.228808,2010-12-01 08:26:00,0.773462,17850.0,20.34,0.562827,4.996809,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,0.066298,2010-12-01 08:26:00,0.359494,17850.0,22.00,0.688442,4.996809,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,-0.228808,2010-12-01 08:26:00,0.773462,17850.0,20.34,0.562827,4.996809,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,-0.228808,2010-12-01 08:26:00,0.773462,17850.0,20.34,0.562827,4.996809,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


# Exploratory Data Analysis

    - USE dataEda for the Exploratory

In [100]:
dataEda.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,customer lifetime value,Total,Recency,...,Category_BAG,Category_CANDLE,Category_CHRISTMAS,Category_CUP,Category_HANGER,Category_MUG,Category_OTHER,Category_PARTY,Category_STICKER,Category_TOY
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,15.30,15.30,372,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,20.34,20.34,372,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,22.00,22.00,372,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,20.34,20.34,372,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,20.34,20.34,372,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
